In [15]:
# Loading the data
import json
import pandas as pd
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from sklearn.covariance import MinCovDet
import numpy as np

# Define the project root (one folder up from /workshop)
project_root = Path("..").resolve()

# Add the project root to sys.path so Jupyter can find your 'src' module
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Load data/valencia_cf_1/matches.json
matches_path = project_root / "tests" / "fixtures" / "testing_data" / "data" / "valencia_cf_1" / "matches.json"
raw_matches = json.load(open(matches_path))
print(f"Loaded {len(raw_matches)} matches from {matches_path}")

Loaded 100 matches from C:\Users\kazik\projects\Gaffers-Clipboard\tests\fixtures\testing_data\data\valencia_cf_1\matches.json


In [16]:
normalized_df = pd.json_normalize(
    raw_matches,
    record_path="player_performances",
    meta=[
        "id", ["data", "half_length"], 
        ["data", "home_team_name"], 
        ["data", "away_team_name"], 
        ["data", "home_stats", "possession"], 
        ["data", "away_stats", "possession"],
        ["data", "home_stats", "xg"],
        ["data", "away_stats", "xg"],
    ]
)

normalized_df["team_possession"] = float("nan")
normalized_df["team_xg"] = float("nan")

for idx, row in normalized_df.iterrows():
    if row["data.home_team_name"] == "Valencia CF":
        normalized_df.at[idx, "team_possession"] = row["data.home_stats.possession"]
        normalized_df.at[idx, "team_xg"] = row["data.home_stats.xg"]
    else:
        normalized_df.at[idx, "team_possession"] = row["data.away_stats.possession"]
        normalized_df.at[idx, "team_xg"] = row["data.away_stats.xg"]

normalized_df = normalized_df.drop(columns=[
    "data.home_team_name", "data.away_team_name", "data.home_stats.possession", "data.away_stats.possession", "data.home_stats.xg", "data.away_stats.xg"])

normalized_df = normalized_df.rename(columns={
    "data.half_length": "half_length",
    "id": "match_id",
})

normalized_df = normalized_df[normalized_df["performance_type"] != "GK"]

normalized_df = normalized_df.dropna(axis=1, how="all")

pos_df = normalized_df[
    normalized_df["positions_played"].apply(lambda x: x in (["RW"], ["LW"]))
]

pos_df = pos_df.dropna(axis=1, how="all")

pos_df = pos_df.drop(columns=["performance_type", "positions_played"])

cols = pos_df.columns.tolist()
cols.insert(0, cols.pop(cols.index("match_id")))
pos_df = pos_df[cols]

print(f"Number of rows: {pos_df.shape[0]}")
print(f"Number of columns: {pos_df.shape[1]}")
print("Columns:")
for col in cols:
    print(f" - {col}")

# Output the max and min of every column, to get a sense of the range of values
for col in pos_df.columns:
    if col in ["match_id", "player_id"]:
        continue
    if pos_df[col].dtype in ["int64", "float64"]:
        print(f"{col}: min={pos_df[col].min()}, max={pos_df[col].max()}")

Number of rows: 296
Number of columns: 22
Columns:
 - match_id
 - player_id
 - goals
 - assists
 - shots
 - shot_accuracy
 - passes
 - pass_accuracy
 - dribbles
 - dribble_success_rate
 - tackles
 - tackle_success_rate
 - offsides
 - fouls_committed
 - possession_won
 - possession_lost
 - minutes_played
 - distance_covered
 - distance_sprinted
 - half_length
 - team_possession
 - team_xg
goals: min=0.0, max=5.0
assists: min=0.0, max=3.0
shots: min=0.0, max=9.0
shot_accuracy: min=0.0, max=100.0
passes: min=0.0, max=48.0
pass_accuracy: min=0.0, max=100.0
dribbles: min=0.0, max=45.0
dribble_success_rate: min=0.0, max=100.0
tackles: min=0.0, max=10.0
tackle_success_rate: min=0.0, max=100.0
offsides: min=0.0, max=4.0
fouls_committed: min=0.0, max=1.0
possession_won: min=0.0, max=13.0
possession_lost: min=0.0, max=11.0
minutes_played: min=1.0, max=114.0
distance_covered: min=0.1, max=14.5
distance_sprinted: min=0.0, max=8.4
team_possession: min=39.0, max=71.0
team_xg: min=0.7, max=13.2


In [17]:
# 1. Select every column EXCEPT your identifiers
cols_to_convert = pos_df.columns.drop(['match_id', 'player_id'])

# 2. Force them all to numeric in one swoop
pos_df[cols_to_convert] = pos_df[cols_to_convert].apply(pd.to_numeric, errors='coerce')

# Optional: If any weird JSON text like "N/A" got turned into a NaN, 
# you can safely fill them with 0s to keep the math from breaking later.
pos_df[cols_to_convert] = pos_df[cols_to_convert].fillna(0)

In [18]:
# Step 1: Removing players with less than 10 minutes played
pos_df = pos_df[pos_df["minutes_played"] >= 10]

In [19]:
# Step 1: Volume masking
# For each percentage stat, their associated volume attempted stat must reach a certain threshold for the percentage 
# stat to be considered valid. For example, if a player attempted 0 passes, their pass accuracy should not be considered valid,
# and should be set to the mean.
volume_perc_pairs = [
    ("passes", "pass_accuracy"),
    ("shots", "shot_accuracy"),
    ("dribbles", "dribble_success_rate"),
    ("tackles", "tackle_success_rate")
]
volume_masks = {
    "passes": 5,
    "shots": 2,
    "dribbles": 3,
    "tackles": 2
}

for volume_col, perc_col in volume_perc_pairs:
    # apply the mask
    valid = pos_df[pos_df[volume_col] >= volume_masks[volume_col]]
    mean_value = valid[perc_col].mean()
    pos_df[perc_col] = pos_df.apply(
        lambda r: r[perc_col] if r[volume_col] >= volume_masks[volume_col] else mean_value,
        axis=1
    )

In [20]:
# Possession adjustment
# 1. The Softened Attacking & Distribution Tax
atk_cols = ["passes", "dribbles", "shots", "possession_lost"]
for col in atk_cols:
    # np.sqrt softens the penalty so extreme performances aren't crushed
    pos_df[col] = pos_df[col] * np.sqrt(50.0 / pos_df["team_possession"])

# 2. The Softened Defensive Tax
def_cols = ["tackles", "possession_won", "fouls_committed"]
for col in def_cols:
    pos_df[col] = pos_df[col] * np.sqrt(50.0 / (100.0 - pos_df["team_possession"]))

In [21]:
total_goals = normalized_df["goals"].sum()
total_assists = normalized_df["assists"].sum()

cdm_goals = pos_df["goals"].sum()
cdm_assists = pos_df["assists"].sum()

cdm_goal_share = cdm_goals / total_goals
cdm_assist_share = cdm_assists / total_assists

print(f'CDM Goal Share: {cdm_goal_share}')
print(f'CDM Assist Share: {cdm_assist_share}')

CDM Goal Share: 0.35789473684210527
CDM Assist Share: 0.3412698412698413


In [22]:
# ---------------------------------------------------------
# Step 1: Standardize Absolute Volume (The Half-Length Fix)
# ---------------------------------------------------------
cum_columns = ["goals", "assists", "shots", "passes", "dribbles", "tackles", 
               "possession_won", "possession_lost", "fouls_committed", "offsides", 
               "distance_covered", "distance_sprinted"]

h_base = 10.0

for col in cum_columns:
    # We overwrite the raw stat to represent a standard 10-minute half game.
    # This ensures 5-min half games don't corrupt our dataset averages later.
    pos_df[col] = pos_df[col] * (h_base / pos_df["half_length"])


# ---------------------------------------------------------
# Step 2: Set Bayesian Smoothing Parameters
# ---------------------------------------------------------
dummy_minutes = 30.0  # Shrinkage anchor (requires ~a half of football to break away from mean)


# ---------------------------------------------------------
# Step 3: Apply Smoothed Per-90 Scaling
# ---------------------------------------------------------
# A. Rare Stats (Assume 0 for dummy minutes)
rare_cols = ["goals", "assists", "shots"]
for col in rare_cols:
    pos_df[f"{col}_p90"] = (pos_df[col] / (pos_df["minutes_played"] + dummy_minutes)) * 90.0


# B. Volume Stats (Assume dataset average for dummy minutes)
volume_cols = ["passes", "dribbles", "tackles", "possession_won", "possession_lost", 
               "fouls_committed", "offsides", "distance_covered", "distance_sprinted"]

for col in volume_cols:
    # 1. Calculate the true, half-length-adjusted dataset average Per 90
    league_avg_p90 = (pos_df[col].sum() / pos_df["minutes_played"].sum()) * 90.0
    
    # 2. Calculate the dummy stat allocation for 45 minutes
    dummy_stat = league_avg_p90 * (dummy_minutes / 90.0)
    
    # 3. Apply the final smoothed formula
    pos_df[f"{col}_p90"] = ((pos_df[col] + dummy_stat) / (pos_df["minutes_played"] + dummy_minutes)) * 90.0

In [23]:
pos_df["xt_bonus_p90"] = 0.25 * (pos_df["distance_sprinted_p90"] / pos_df["distance_covered_p90"]) * np.log(pos_df["pass_accuracy"] * pos_df["passes_p90"] + 1)

In [24]:
# Columns to log for PCA only; do NOT modify pos_df in-place
cols_to_log = ["goals_p90", "assists_p90", "shots_p90", "offsides_p90", "fouls_committed_p90", "possession_won_p90", "possession_lost_p90"]

In [25]:
negative_stats = ["fouls_committed_p90", "offsides_p90", "possession_lost_p90"]

# Build PCA dataframe from pos_df and apply log to long-tail columns (PCA-only)
pca_df = pos_df.copy()
pca_df[cols_to_log] = np.log(pca_df[cols_to_log] + 1)

# Define ordered stat list used for PCA (must match final weight order)
stat_names = ["goals_p90", "assists_p90", "shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xt_bonus_p90"]
# Calculate z-scores on the log-transformed PCA dataframe
for col in stat_names:
    if col == "goals":
        mean = np.log(pos_df["team_xg"] * st_goal_share + 1)
        std = np.log(pca_df[col]).std()
    elif col == "assists":
        mean = np.log(pos_df["team_xg"] * st_assist_share + 1)
        std = np.log(pca_df[col]).std()
    else:
        mean = pca_df[col].mean()
        std = pca_df[col].std()
    if std == 0:
        pca_df[f"{col}_z"] = 0
    elif col in negative_stats:
        pca_df[f"{col}_z"] = (mean - pca_df[col]) / std
    else:
        pca_df[f"{col}_z"] = (pca_df[col] - mean) / std

# final matrix for PCA (z-scores from the log-transformed matrix)
final_pos_df = pca_df[[f"{col}_z" for col in stat_names]]

In [26]:
blocks = {
    "Attacking": [
        "goals_p90_z", "assists_p90_z", "shots_p90_z", "shot_accuracy_z"
    ],
    "Passing": [
        "passes_p90_z", "pass_accuracy_z"
    ],
    "Dribbling": [
        "dribbles_p90_z", "dribble_success_rate_z"
    ],
    "Safety": [
        "possession_lost_p90_z"
    ],
    "Defending": [
        "tackles_p90_z", "tackle_success_rate_z", "possession_won_p90_z"
    ],
    "Workrate_and_Threat": [
        "distance_covered_p90_z", "distance_sprinted_p90_z", "xt_bonus_p90_z"
    ],
    "Tactical_Discipline": [
        "offsides_p90_z", "fouls_committed_p90_z"
    ],
}

winger_philosophy = {
    "Attacking": 1.8,
    "Passing": 1.6,
    "Dribbling": 2.4,
    "Safety": 0.4,
    "Defending": 0.5,
    "Workrate_and_Threat": 2.0,
    "Tactical_Discipline": 1.0,
}

scaled_pca_df = final_pos_df.copy()

for block_name, stats in blocks.items():
    k = len(stats)  # Number of stats in this block
    scale_factor = np.sqrt(k) / winger_philosophy[block_name]

    for stat in stats:
        # Scale the Z-score down
        scaled_pca_df[stat] = scaled_pca_df[stat] / scale_factor

mincovdet_random_state = 42

mcd = MinCovDet(random_state=mincovdet_random_state, support_fraction=0.98).fit(scaled_pca_df)

cov = mcd.covariance_

eigenvalues, eigenvectors = np.linalg.eigh(cov)

sorted_indicies = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sorted_indicies]
eigenvectors = eigenvectors[:, sorted_indicies]

k = 9

top_eigenvectors = eigenvectors[:, :k]
top_variances = eigenvalues[:k]

weighted_loadings = np.abs(top_eigenvectors) * np.sqrt(top_variances)

raw_weights = np.sum(weighted_loadings, axis=1)

final_weights = raw_weights / np.sum(raw_weights)

stat_names = ["goals", "assists", "shots", "shot_accuracy", "passes", "pass_accuracy",
              "dribbles", "dribble_success_rate", "tackles", "tackle_success_rate", "offsides",
              "fouls_committed", "possession_won", "possession_lost", "distance_covered", "distance_sprinted", "xt_bonus"]

print("\nFinal weights (normalized to sum to 1):")
for stat, weight in zip(stat_names, final_weights):
    # print to 4 dp
    print(f"{stat}: {weight:.4f}")


Final weights (normalized to sum to 1):
goals: 0.0684
assists: 0.0672
shots: 0.0726
shot_accuracy: 0.0726
passes: 0.0891
pass_accuracy: 0.0631
dribbles: 0.1158
dribble_success_rate: 0.1063
tackles: 0.0075
tackle_success_rate: 0.0068
offsides: 0.0444
fouls_committed: 0.0355
possession_won: 0.0093
possession_lost: 0.0262
distance_covered: 0.0728
distance_sprinted: 0.0727
xt_bonus: 0.0697


In [27]:
# Outputting the final weights to a shared json file in the project root called position_weights.json
section_names = ["RW", "LW"]
col_names = ["goals_p90", "assists_p90", "shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xt_bonus_p90"]
weights_dict = dict(zip(col_names, final_weights))
position_weights_path = project_root / "workshop" / "position_weights.json"

if position_weights_path.exists():
    with open(position_weights_path, "r") as f:
        position_weights = json.load(f)
else:
    position_weights = {}

for section_name in section_names:
    position_weights[section_name] = weights_dict

with open(position_weights_path, "w") as f:
    json.dump(position_weights, f, indent=4)

In [28]:
# Now lets look at storing the means and standard deviations used for the z-score calculations in a shared json file called position_means_stds.json
# Each position writes its baseline stats into its own section so the rating notebooks can read the same shared calibration data.
section_names = ["RW", "LW"]
means_stds = {}
for col in col_names:
    mean = pos_df[col].mean()
    std = pos_df[col].std()
    means_stds[col] = {"mean": mean, "std": std}
position_means_stds_path = project_root / "workshop" / "position_means_stds.json"

if position_means_stds_path.exists():
    with open(position_means_stds_path, "r") as f:
        position_means_stds = json.load(f)
else:
    position_means_stds = {}

for section_name in section_names:
    position_means_stds[section_name] = means_stds

with open(position_means_stds_path, "w") as f:
    json.dump(position_means_stds, f, indent=4)